# OrionBelt Semantic Layer — Quickstart

This notebook demonstrates OrionBelt's semantic layer using the **TPC-H** benchmark dataset
in DuckDB. Everything runs locally — no cloud database needed.

**What you'll see:**

1. Create a TPC-H database (DuckDB, ~60K rows)
2. Explore dimensions, measures, and metrics
3. Compile OBML queries to SQL across 5 dialects
4. Execute queries and see real results
5. Multi-fact CFL (Composite Fact Layer) in action
6. Lineage, ER diagrams, and search

## Step 0: Create the TPC-H Database

DuckDB has a built-in TPC-H data generator. This creates 8 tables with realistic
supply-chain data (regions, nations, customers, orders, line items, parts, suppliers).

In [ ]:
import duckdb

db_path = "tpch.duckdb"
con = duckdb.connect(db_path)
con.execute("INSTALL tpch")
con.execute("LOAD tpch")
con.execute("CALL dbgen(sf=0.01)")

# Show what we got
tables = con.execute(
    "SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name"
).fetchall()
for name, size in tables:
    print(f"  {name:12s}  {size:>8,} rows")
con.close()
print(f"\nDatabase saved to {db_path}")

## Step 1: Start the API

Run this in a **separate terminal** from the `examples/` directory:

```bash
# From the repo root:
FLIGHT_ENABLED=true \
  DB_VENDOR=duckdb \
  DUCKDB_DATABASE=examples/tpch.duckdb \
  MODEL_FILE=examples/tpch.obml.yml \
  uv run orionbelt-api
```

The `MODEL_FILE` flag enables **single-model mode** — the model is auto-loaded and
shortcut endpoints (`/v1/query/sql`, `/v1/query/execute`, etc.) work without session management.

`FLIGHT_ENABLED=true` enables the `/v1/query/execute` endpoint for running queries against the database.

In [ ]:
import json
from urllib.request import Request, urlopen

API = "http://localhost:8000"


def api(method: str, path: str, body: dict | None = None) -> dict | list | str:
    """Minimal HTTP helper — stdlib only, no pip install needed."""
    url = f"{API}{path}"
    data = json.dumps(body).encode() if body is not None else None
    headers = {"Content-Type": "application/json"} if data else {}
    req = Request(url, data=data, headers=headers, method=method)
    with urlopen(req) as resp:
        text = resp.read().decode()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text


# Verify the API is running
health = api("GET", "/health")
print(f"OrionBelt API v{health['version']} is running")

## Step 2: Explore the Model

The TPC-H model defines 12 dimensions, 9 measures, and 3 metrics.
All shortcut endpoints auto-resolve to the loaded model.

In [ ]:
# Dimensions — the business concepts you can group by
dims = api("GET", "/v1/dimensions")
print("Dimensions:")
for d in dims:
    print(f"  {d['label']:20s}  ({d['dataObject']}.{d['column']})")

In [ ]:
# Measures — aggregations over fact table columns
print("Measures:")
for m in api("GET", "/v1/measures"):
    agg = m.get("aggregation", "")
    expr = m.get("expression", "")
    detail = expr if expr else agg
    print(f"  {m['label']:20s}  {detail}")

In [ ]:
# Metrics — cross-measure formulas
print("Metrics:")
for m in api("GET", "/v1/metrics"):
    print(f"  {m['label']:20s}  = {m['expression']}")

## Step 3: Compile Queries

Write queries using business names — OrionBelt compiles them to optimized,
dialect-specific SQL with automatic joins.

### Star schema: Revenue by Region

In [ ]:
result = api("POST", "/v1/query/sql", {
    "query": {
        "select": {
            "dimensions": ["Region"],
            "measures": ["Revenue", "Order Count"],
        },
    },
    "dialect": "duckdb",
})
print(result["sql"])

OrionBelt automatically resolves the join path:
`Line Item → Orders → Customer → Nation → Region`

### Same query, 5 dialects

Each dialect adapts identifier quoting, functions, and SQL syntax:

In [ ]:
query = {
    "query": {
        "select": {
            "dimensions": ["Brand", "Ship Mode"],
            "measures": ["Revenue", "Total Quantity"],
        },
        "limit": 5,
    },
}

for dialect in ["duckdb", "postgres", "snowflake", "bigquery", "clickhouse"]:
    result = api("POST", "/v1/query/sql", {**query, "dialect": dialect})
    lines = result["sql"].strip().split("\n")
    # Show first 3 lines to highlight quoting differences
    print(f"\n-- {dialect.upper()}")
    for line in lines[:3]:
        print(f"  {line}")
    if len(lines) > 3:
        print(f"  ... ({len(lines)} lines total)")

## Step 4: Execute Queries (Real Data!)

The `/v1/query/execute` endpoint compiles OBML and runs the SQL against the database,
returning actual results.

In [ ]:
result = api("POST", "/v1/query/execute", {
    "query": {
        "select": {
            "dimensions": ["Region"],
            "measures": ["Revenue", "Order Count", "Total Quantity"],
        },
        "orderBy": [{"name": "Revenue", "direction": "desc"}],
    },
    "dialect": "duckdb",
})

# Display as a formatted table
cols = result["columns"]
rows = result["rows"]
widths = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
header = "  ".join(f"{c:<{widths[i]}}" for i, c in enumerate(cols))
print(header)
print("-" * len(header))
for row in rows:
    print("  ".join(f"{str(v):<{widths[i]}}" for i, v in enumerate(row)))
print(f"\n{result['row_count']} rows")

In [ ]:
# Revenue by Nation and Market Segment (top 10)
result = api("POST", "/v1/query/execute", {
    "query": {
        "select": {
            "dimensions": ["Nation", "Market Segment"],
            "measures": ["Revenue"],
        },
        "orderBy": [{"name": "Revenue", "direction": "desc"}],
        "limit": 10,
    },
    "dialect": "duckdb",
})

cols = result["columns"]
rows = result["rows"]
widths = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
header = "  ".join(f"{c:<{widths[i]}}" for i, c in enumerate(cols))
print(header)
print("-" * len(header))
for row in rows:
    print("  ".join(f"{str(v):<{widths[i]}}" for i, v in enumerate(row)))

### Filters: WHERE + HAVING + ORDER BY

In [ ]:
# High-value urgent orders by ship mode
result = api("POST", "/v1/query/execute", {
    "query": {
        "select": {
            "dimensions": ["Ship Mode"],
            "measures": ["Revenue", "Order Count"],
        },
        "where": [
            {"dimension": "Order Priority", "operator": "=", "value": "1-URGENT"},
        ],
        "having": [
            {"measure": "Order Count", "operator": ">", "value": 100},
        ],
        "orderBy": [{"name": "Revenue", "direction": "desc"}],
    },
    "dialect": "duckdb",
})

print("Urgent orders by ship mode (count > 100):\n")
print(f"{'Ship Mode':15s}  {'Revenue':>15s}  {'Orders':>8s}")
print("-" * 42)
for row in result["rows"]:
    print(f"{row[0]:15s}  {row[1]:>15,.2f}  {row[2]:>8,}")

## Step 5: Multi-Fact CFL (Composite Fact Layer)

When a query references measures from **independent fact tables**, OrionBelt
automatically uses the CFL planner. It generates `UNION ALL` with NULL padding
to prevent fan-trap row multiplication.

In this model, `Line Item` and `Part Supply` are independent facts — both join to
`Part` and `Supplier`, but neither joins to the other.

In [ ]:
# Revenue (from Line Item) + Supply Cost (from Part Supply) by Brand → CFL!
result = api("POST", "/v1/query/sql", {
    "query": {
        "select": {
            "dimensions": ["Brand"],
            "measures": ["Revenue", "Total Supply Cost"],
        },
        "limit": 10,
    },
    "dialect": "duckdb",
})
print("--- Generated SQL (CFL with UNION ALL) ---\n")
print(result["sql"])

In [ ]:
# Execute the multi-fact query
result = api("POST", "/v1/query/execute", {
    "query": {
        "select": {
            "dimensions": ["Brand"],
            "measures": ["Revenue", "Total Supply Cost"],
        },
        "orderBy": [{"name": "Revenue", "direction": "desc"}],
        "limit": 10,
    },
    "dialect": "duckdb",
})

print(f"{'Brand':12s}  {'Revenue':>15s}  {'Supply Cost':>15s}")
print("-" * 46)
for row in result["rows"]:
    print(f"{row[0]:12s}  {row[1]:>15,.2f}  {row[2]:>15,.2f}")

## Step 6: Metrics (Cross-Measure Formulas)

Metrics are calculated from measures: `{[Total Discount]} / {[Gross Amount]}`.
OrionBelt resolves the formula and generates the correct SQL expression.

In [ ]:
result = api("POST", "/v1/query/execute", {
    "query": {
        "select": {
            "dimensions": ["Market Segment"],
            "measures": ["Revenue", "Gross Amount", "Discount Rate", "Average Order Value"],
        },
        "orderBy": [{"name": "Revenue", "direction": "desc"}],
    },
    "dialect": "duckdb",
})

print(f"{'Segment':15s}  {'Revenue':>14s}  {'Gross':>14s}  {'Disc%':>8s}  {'Avg Order':>12s}")
print("-" * 70)
for row in result["rows"]:
    print(f"{row[0]:15s}  {row[1]:>14,.2f}  {row[2]:>14,.2f}  {row[3]:>7.2%}  {row[4]:>12,.2f}")

## Step 7: Lineage & Search

In [ ]:
# Explain how "Revenue" maps to physical tables and columns
explain = api("GET", "/v1/explain/Revenue")
print(json.dumps(explain, indent=2))

In [ ]:
# Search for anything related to "order"
results = api("POST", "/v1/find", {"query": "order"})
for item in results:
    print(f"  [{item['type']:10s}]  {item['label']}")

## Step 8: ER Diagram

Generate a Mermaid ER diagram showing the model's data objects, columns, and joins.

In [ ]:
# Discover session/model IDs (auto-created by MODEL_FILE)
sessions = api("GET", "/v1/sessions")
sid = sessions[0]["session_id"]
models = api("GET", f"/v1/sessions/{sid}/models")
mid = models[0]["model_id"]

er = api("GET", f"/v1/sessions/{sid}/models/{mid}/diagram/er")
print(er["mermaid"])

Copy the Mermaid output above into the [Mermaid Live Editor](https://mermaid.live)
to see the diagram visually, or view it in the **Gradio UI** which renders it automatically.

---

## Next Steps

- **Gradio UI** — Visual model explorer with ER diagrams, SQL preview, and query builder:
  ```bash
  docker pull ralforion/orionbelt-ui
  docker run -p 7860:7860 -e API_BASE_URL=http://host.docker.internal:8080 ralforion/orionbelt-ui
  ```
- **Arrow Flight SQL** — Connect DBeaver, Tableau, or Power BI directly — see [docs/drivers.md](../docs/drivers.md)
- **DB-API 2.0 Drivers** — Use `ob-duckdb`, `ob-postgres`, `ob-snowflake`, etc. from Python
- **MCP Server** — Connect AI assistants via [orionbelt-semantic-layer-mcp](https://github.com/ralfbecher/orionbelt-semantic-layer-mcp)
- **Full documentation** — [ralforion.com/orionbelt-semantic-layer](https://ralforion.com/orionbelt-semantic-layer/)